# 5.16 配比怎麼落到實際 batch？


有90筆短答，每筆含1個有效答案 token；另有10筆長答，每筆含20個。這裡每個有效答案位置算一個 token。按答案筆數，短答是90/100，佔九成；按有效答案 token 數，短答是90、長答是200，總共290。若把全部290個位置的預測代價加總，再除以290來算平均 loss，長答的計分位置權重就是200/290，約七成。這是位置的占比，各來源的實際代價貢獻還取決於那些位置猜錯多少。配比先說清楚是筆數還是 token 數。

source標記資料來源或任務。以下用固定長度手算，真batch則逐筆數label不為-100的位置。PAD是把不同長度的資料補齊時使用的佔位符號；問題與這些補格都不計監督分母。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
counts = {"短答": 90, "長答": 10}
lengths = {"短答": 1, "長答": 20}
tokens = {}
for source in counts:
    tokens[source] = counts[source] * lengths[source]
total = sum(tokens.values())
ratios = {source: amount / total for source, amount in tokens.items()}
print("有效答案token數", tokens)
print("有效答案token比例", ratios)

輸入counts是筆數、lengths是假設每筆有效答案token數。迴圈逐來源相乘，得到短答90、長答200，總共290。輸出比例約0.310與0.690，所以少數長答貢獻約七成監督。這裡只計答案位置，不把用戶問題、填充或被忽略的文字混進去；真正batch長度不齊時，要逐筆數有效label，而不是用補齊後的總格數。

如果目標真的是token比例90:10，需要調整抽樣筆數，或每筆取多少有效內容。假設兩類固定長度1與20，要令長答token只佔一成，短答筆數約應為長答筆數的180倍。你能從 `短數×1 : 長數×20 = 9:1` 算出這個比例，但數據與batch實際可行性還需檢查。

若有效答案token比例要90:10，這兩種長度需短答筆數：長答筆數約180:1。也可以改來源loss權重，但那是計分貢獻，實際答案量仍不同。截斷改token數、重採樣改筆數；最好累計一段訓練並分來源評估，避免總平均遮住少數任務退步。

練習只把長答長度20改成1，先預測兩類token為90與10、比例回到0.9與0.1，再跑核對。筆數完全沒變，變化來自答案長度。再恢復20並把長答筆數改成5，手算長答100、總190、比例約0.526，驗證少一半長答仍可能主導監督。

<details>
<summary>補充：實作約定與原始紀錄</summary>

這個差別已在[7.16的回放實驗](https://birdhackor.github.io/tiny-perceptron-vlm/7.16.html)實際出現：兩支都更新500次、每批16題，只練加法的累計有效答案是18,453個，混回較長屬性答案的支線卻有36,384個。配方的筆數與步數相同，監督總量仍差近一倍。因此比較資料混合成績時，要把實際計分位置帶回來，不能只寫「都跑500步」。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
